# Jupyter 환경에서 흉부 X선(CXR) 이미지 로드

**흐름:** RStudio(R) 에서 만든 결핵 코호트 `tb_cohort.csv` 를 읽어서,
그 코호트가 가리키는 흉부 X선(CXR) 인스턴스를 불러와 **이미지로 렌더링**한다.

CSV 의 컬럼은 `subject_id, study_id, local_path` 이고,
`local_path` 는 CXR DICOM 이 저장된 GCS 경로(`gs://...`)다.

In [ ]:
# 필요한 패키지 (없으면 설치)
# pydicom: DICOM(.dcm) 영상을 읽는 라이브러리
try:
    import pydicom
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "pydicom", "pylibjpeg", "pylibjpeg-libjpeg", "tqdm"])
    import pydicom

import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

In [ ]:
# 레포 루트 자동 탐색 (check_feasibility 폴더가 보일 때까지 상위로 올라감)
def find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "check_feasibility").exists():
            return cand
    raise RuntimeError("레포 루트를 찾지 못했습니다. ROOT 를 직접 지정하세요.")

ROOT = find_root(Path.cwd())
print("레포 루트:", ROOT)

COHORT_CSV = ROOT / "practice" / "outputs" / "tb_cohort.csv"
assert COHORT_CSV.exists(), f"먼저 RStudio에서 02_build_tb_cohort.R 를 실행하세요: {COHORT_CSV}"

# CXR DICOM 을 내려받아 둘 로컬 캐시 폴더
CACHE_DIR = ROOT / "practice" / "outputs" / "cxr_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

## 1. RStudio 가 만든 결핵 코호트 불러오기

In [ ]:
cohort = pd.read_csv(COHORT_CSV, dtype={"subject_id": str, "study_id": str})
print(f"결핵 코호트: 환자 {cohort.subject_id.nunique()}명 / CXR {len(cohort)}장")

cohort.head(8).style.set_caption("tb_cohort.csv (RStudio 산출물)").hide(axis="index")

## 2. local_path(S3) 에서 CXR DICOM 내려받기

CXR 원본은 S3에 있다. 코호트에서 몇 장만 골라 로컬 캐시로 내려받는다.
(전체를 받지 않고 시연용으로 앞의 몇 장만 사용한다.)

In [ ]:
N_SHOW = 4  # 시연용으로 몇 장만

def fetch_dicom(s3_uri: str) -> Path:
    """local_path(S3 URI) 를 로컬 캐시로 내려받고 로컬 경로를 반환."""
    dst = CACHE_DIR / Path(s3_uri).name
    if not dst.exists():
        subprocess.run(["aws", "s3", "cp", s3_uri, str(dst)],
                       check=True, capture_output=True)
    return dst

records = cohort.head(N_SHOW).to_dict("records")
for r in tqdm(records, desc="CXR 다운로드"):
    r["local_file"] = fetch_dicom(r["local_path"])
print("다운로드 완료:", CACHE_DIR)

## 3. pydicom 으로 로드 후 흉부 X선 이미지로 그리기

In [ ]:
def read_cxr(dcm_path: Path) -> np.ndarray:
    """DICOM 을 읽어 0~1 정규화된 흑백 영상 배열로 반환."""
    ds = pydicom.dcmread(str(dcm_path))
    arr = ds.pixel_array.astype(np.float32)
    # MONOCHROME1 은 흑백이 반전되어 있으므로 뒤집는다
    if ds.get("PhotometricInterpretation") == "MONOCHROME1":
        arr = arr.max() - arr
    arr = (arr - arr.min()) / (np.ptp(arr) + 1e-9)
    return arr

n = len(records)
fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 4))
axes = np.atleast_1d(axes)
for ax, r in zip(axes, records):
    img = read_cxr(r["local_file"])
    ax.imshow(img, cmap="gray")
    ax.set_title(f"subject={r['subject_id']}\nstudy={r['study_id']}", fontsize=9)
    ax.axis("off")
# 그림 제목은 한글 폰트가 없어도 깨지지 않도록 ASCII 로 둔다
fig.suptitle("TB cohort chest X-ray (CXR)", fontsize=12)
fig.tight_layout()
plt.show()

## 정리

- RStudio: `DatabaseConnector` 로 **CDM DB(PostgreSQL)** 에 접속 (실제 병원 환경과 동일)
- RStudio: `condition_occurrence` 로 **결핵 코호트** 구성 후
  `image_occurrence` 의 CXR `local_path` 를 붙여 `tb_cohort.csv` 저장
- Jupyter: 그 CSV 의 `local_path`(GCS) 로 **CXR 이미지를 내려받아 렌더링**

`subject_id, study_id, local_path` 세 컬럼짜리 CSV 하나가
두 실습 환경을 이어 주는 다리 역할을 했다.

> `image_occurrence` 의 `local_path` 는 데이터를 만든 서버 기준 경로(`s3://...`)였다.
> 02 에서 접두사를 `gs://` 로 바꿔 저장했기 때문에 여기서 바로 읽을 수 있다.